# Exercises

There are three exercises in this notebook:

1. Use the cross-validation method to test the linear regression with different $\alpha$ values, at least three.
2. Implement a SGD method that will train the Lasso regression for 10 epochs.
3. Extend the Fisher's classifier to work with two features. Use the class as the $y$.

## 1. Cross-validation linear regression

You need to change the variable ``alpha`` to be a list of alphas. Next do a loop and finally compare the results.

In [16]:
import numpy as np

x = np.array([188, 181, 197, 168, 167, 187, 178, 194, 140, 176, 168, 192, 173, 142, 176]).reshape(-1, 1).reshape(15,1)
y = np.array([141, 106, 149, 59, 79, 136, 65, 136, 52, 87, 115, 140, 82, 69, 121]).reshape(-1, 1).reshape(15,1)

x = np.asmatrix(np.c_[np.ones((15,1)),x])

I = np.identity(2)
alpha = [0.01, 0.1, 1.0, 10.0]

results = []
for a in alpha:
    w = np.linalg.inv(x.T*x + a * I)*x.T*y
    w=w.ravel()

    y_pred = np.array(x * np.asmatrix(w).T)
    mse = np.mean((y - y_pred)**2)
    results.append((a, mse))

for a, mse in results:
    print(f"Alpha: {a}, MSE: {mse:.4f}")


Alpha: 0.01, MSE: 373.7938
Alpha: 0.1, MSE: 426.0451
Alpha: 1.0, MSE: 592.4636
Alpha: 10.0, MSE: 645.5800


## 2. Implement based on the Ridge regression example, the Lasso regression.

Please implement the SGD method and compare the results with the sklearn Lasso regression results. 

In [43]:
def sgd(x, y, alpha=0.1, lr=1e-4, epochs=10):
    n_samples, n_features = x.shape

    w = np.zeros((n_features, 1))

    for epoch in range(epochs):
        for i in range(n_samples):
            xi = x[i:i+1].T
            yi = y[i, 0]
            
            y_pred = (w.T @ xi).item()
            error = y_pred - yi
            
            grad = xi * error
            
            for j in range(1, n_features):
                grad[j] += alpha * np.sign(w[j])

            w = w - lr * grad
    
    return w

In [44]:
x = np.array([188, 181, 197, 168, 167, 187, 178, 194, 140, 176, 168, 192, 173, 142, 176]).reshape(-1, 1).reshape(15,1)
y = np.array([141, 106, 149, 59, 79, 136, 65, 136, 52, 87, 115, 140, 82, 69, 121]).reshape(-1, 1).reshape(15,1)

x = np.asmatrix(np.c_[np.ones((15,1)),x])

I = np.identity(2)
alpha = 0.1 


w = sgd(x, y, alpha)
w=w.ravel()


In [46]:
from sklearn.linear_model import Lasso

model = Lasso(alpha=0.1, fit_intercept=False, max_iter=10000)
x_array = np.asarray(x)
y_array = np.asarray(y)
model.fit(x_array, y_array)

print("SGD:", w)
print("Sklearn:", model.coef_)

SGD: [[-8.41159492e+42 -1.77429254e+45]]
Sklearn: [-168.19471366    1.54607814]


## 3. Extend the Fisher's classifier

Please extend the targets of the ``iris_data`` variable and use it as the $y$.

In [50]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris

iris_data = load_iris()
iris_df = pd.DataFrame(iris_data.data,columns=iris_data.feature_names)
iris_df.head()

x = iris_df[['sepal width (cm)', 'sepal length (cm)']].values 
y = iris_data.target

dataset_size = np.size(x)

# mean_x, mean_y = np.mean(x), np.mean(y)

# SS_xy = np.sum(y * x) - dataset_size * mean_y * mean_x
# SS_xx = np.sum(x * x) - dataset_size * mean_x * mean_x

# a = SS_xy / SS_xx
# b = mean_y - a * mean_x


# y_pred = a * x + b

n = len(y)

# Means
mean_x1 = np.mean(x1)
mean_x2 = np.mean(x2)
mean_y  = np.mean(y)

# Coefficients via normal equations (multivariate least-squares Fisher)
SS_x1y  = np.sum(y * x1) - n * mean_y * mean_x1
SS_x2y  = np.sum(y * x2) - n * mean_y * mean_x2
SS_x1x1 = np.sum(x1 * x1) - n * mean_x1 ** 2
SS_x2x2 = np.sum(x2 * x2) - n * mean_x2 ** 2
SS_x1x2 = np.sum(x1 * x2) - n * mean_x1 * mean_x2

# Solve 2x2 system: [[SS_x1x1, SS_x1x2],[SS_x1x2, SS_x2x2]] * [a1,a2] = [SS_x1y, SS_x2y]
A = np.array([[SS_x1x1, SS_x1x2],
              [SS_x1x2, SS_x2x2]])
b_vec = np.array([SS_x1y, SS_x2y])
a1, a2 = np.linalg.solve(A, b_vec)
b = mean_y - a1 * mean_x1 - a2 * mean_x2

y_pred = a1 * x1 + a2 * x2 + b
print(f"a1={a1:.4f}, a2={a2:.4f}, b={b:.4f}")
print(f"MSE: {np.mean((y - y_pred)**2):.4f}")

# Visualize
plt.figure(figsize=(8, 5))
scatter = plt.scatter(x1, x2, c=y, cmap='viridis', edgecolors='k', s=60)
plt.colorbar(scatter, label='Class (0=setosa, 1=versicolor, 2=virginica)')
plt.xlabel('Sepal Width (cm)')
plt.ylabel('Petal Length (cm)')
plt.title("Fisher's Classifier — Two Features")
plt.tight_layout()
plt.show()


AttributeError: module 'matplotlib._api' has no attribute 'MatplotlibDeprecationWarning'